# Lección 6: Analítica y visualización

Práctica individual de la Lección 6 (run-only). Modelamos sobre agregados GEIH (lineal y un MLP pequeño), hacemos gráficos ligados a una pregunta de decisión y discutimos la gobernanza al presentar evidencia. Código evaluable en week-3-group Parte B.

---
<font size="3">
Christian Cabrera Jojoa<br>
Assistant Research Professor<br>
Department of Computer Science and Technology<br>
University of Cambridge<br>
chc79@cam.ac.uk
</font>

---
**Curso:** Big Data<br>
**Departamento del curso:** Departamento de Matemáticas y Estadística - Facultad de Ciencias Exactas y Naturales<br>
**Institución del curso:** Universidad de Nariño


## Repaso de Python para este cuaderno

En esta lección **modelamos y visualizamos** para responder una pregunta de decisión. Lo nuevo frente a las lecciones anteriores son tres ideas de aprendizaje automático con **`scikit-learn`**: separar datos, entrenar un modelo y medir qué tan bien predice. Trabajamos **siempre sobre agregados** (departamento × mes), **nunca** sobre microdatos de personas ni texto de noticias.

### Separar datos en entrenamiento, validación y prueba

Para saber si un modelo **generaliza** (no solo memoriza) usamos **tres** subconjuntos:

- **Entrenamiento** (*train*): el modelo **aprende** con estos datos.
- **Validación** (*validation*): comparamos modelos y **elegimos** el mejor (sin tocar la prueba).
- **Prueba** (*test*): la evaluación **final e imparcial**, con datos que nunca se usaron para decidir.

`train_test_split` separa en dos; para obtener tres, lo aplicamos **dos veces**: primero apartamos la prueba, luego dividimos el resto en entrenamiento y validación.

In [ ]:
from sklearn.model_selection import train_test_split

# Datos de juguete: x va de 1 a 8, y es 10 veces x
X = [[1], [2], [3], [4], [5], [6], [7], [8]]
y = [10, 20, 30, 40, 50, 60, 70, 80]

# 1) Apartamos el 25 % para la prueba final
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
# 2) Del resto, apartamos un tercio para validación
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.33, random_state=42)

print("Entrenamiento:", len(X_train), "| Validación:", len(X_val), "| Prueba:", len(X_test))
assert len(X_test) == 2 and len(X_val) >= 1
print("train/val/test: OK")

### Entrenar una regresión lineal

Un modelo **lineal** busca la recta que mejor relaciona la entrada con la salida. **`fit`** aprende; **`predict`** estima para datos nuevos.

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit([[1], [2], [3]], [2, 4, 6])  # aprende la relación y = 2x
pred = modelo.predict([[4]])[0]          # estima para x = 4
print("Predicción para x=4:", pred)
assert abs(pred - 8.0) < 0.01
print("LinearRegression: OK")

### Medir el error: R² y MAE

Necesitamos números para comparar modelos:

- **R²** (entre 0 y 1): qué fracción de la variación explica el modelo. Más alto, mejor.
- **MAE** (error absoluto medio): en promedio, cuánto se equivoca, en las unidades de `y`. Más bajo, mejor.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

real = [10, 20, 30]
estimado = [11, 19, 32]
print("R²:", round(r2_score(real, estimado), 3))
print("MAE:", round(mean_absolute_error(real, estimado), 3))
print("Métricas: OK")

---

## Instrucciones

En la Lección 5 trajimos datos: una capa **`curated/`** de agregados GEIH (oficial) y, opcionalmente, un `staging/news_labor.parquet` de noticias (no oficial). Ya tenemos evidencia falta **convertirla en una respuesta para quien decide**.

**Qué hacemos en esta lección.** Pasamos de *tener datos* a *informar una decisión*. La pregunta guía es:

> *Como analista del mercado laboral, ¿qué evidencia le mostraría esta semana a quien toma decisiones?*

Para responderla: (1) ajustamos un **modelo** sobre los agregados GEIH, (2) lo comparamos con un **modelo más flexible** (una red neuronal pequeña) y analizamos un dilema real: **precisión vs. poder explicar**, (3) graficamos la **tasa del stream** como contexto, y (4) reunimos todo en un **tablero** rotulado por capa.

**Una regla de gobernanza, desde el inicio.** Modelamos **solo sobre agregados** (departamento × mes), nunca sobre datos de personas individuales ni sobre el texto de las noticias. Las noticias podrían ser un **monitor de discurso**, no una estadística.

Antes de esta práctica se debe ejecutar **`l5-ingestion`** y tener (o simular) `data/curated/` y, opcional, `staging/news_labor.parquet`.

Las tres capas de evidencia que aparecen en esta lección:

| Capa | Fuente | Para qué la usamos |
|------|--------|--------------------|
| **Oficial** | Agregados GEIH en `curated/` | Modelo y gráfico principal |
| **Macro (opcional)** | TRM / COLCAP en `staging/` | Gráfico de contexto |
| **Medios (opcional)** | `news_labor.parquet` | Monitor — **no es dato oficial** |

**Qué hace el cuaderno (en orden).**

| Parte | Tema |
|-------|------|
| **1** | Montar Drive, rutas a `curated/` e instalar librerías |
| **2** | Modelo **lineal** sobre agregados + gráfico (capa oficial) |
| **3** | **Red neuronal pequeña**: comparar y **elegir** modelo (validación → prueba) |
| **4** | **Monitor del stream**: tasa de titulares en el tiempo (capa no oficial) |
| **5** | **Tablero** que reúne las capas de evidencia |

Ver **Tareas** al final.

---

## Parte 1. Configuración: Drive, rutas y librerías

Montamos la **misma raíz** que en las lecciones 3 a 5, para leer la capa `curated/` que dejó la ingesta.

In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True

# Detectamos Colab para montar Drive solo allí
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/udenar/cease/2026/big-data/")
    print(f"Raíz en Drive: {WORK_ROOT}")
else:
    WORK_ROOT = Path(".")
    print(f"Raíz local: {WORK_ROOT.resolve()}")

Definimos rutas. Leemos de `curated/` (oficial) y, si existe, de `staging/` (noticias). Guardamos los gráficos en `outputs/`.

In [ ]:
CURATED_DIR = WORK_ROOT / "data" / "curated"          # agregados oficiales (entrada)
STAGING_DIR = WORK_ROOT / "data" / "staging"          # noticias (opcional)
SCHEMA_PATH = WORK_ROOT / "data" / "schema_contract.json"
MANIFEST_PATH = WORK_ROOT / "manifest.json"
OUTPUTS_DIR = WORK_ROOT / "outputs"                   # aquí guardamos figuras
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Buscamos los agregados que dejó la Lección 5 / Parte A
curated_files = sorted(CURATED_DIR.glob("geih_*.parquet"))
print("Archivos curated encontrados:", len(curated_files))

Instalamos analítica y visualización. **scikit-learn** para los modelos; **Plotly** y **matplotlib** para graficar.

In [ ]:
%pip install -q polars pyarrow scikit-learn plotly matplotlib

In [ ]:
import json
from datetime import datetime, timedelta, timezone

import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

print("Dependencias: OK")

**Comprobar:**

In [ ]:
assert WORK_ROOT.is_dir()
assert len(curated_files) >= 1, (
    "No hay agregados en curated/. Ejecute l5-ingestion (o week-3-group Parte A) primero."
)
print("Parte 1, configuración: OK")

---

## Parte 2. Modelar sobre agregados oficiales

Tomamos los agregados GEIH y ajustamos un modelo simple: predecir el empleo ponderado a partir del **mes** y el **año**. Este es un ejemplo **didáctico** y no causal, sirve para ver el flujo entrenar → predecir → medir, y para tener una primera lectura de tendencia.

### Paso 2.1. Cargar y preparar la tabla

In [ ]:
# Leemos el primer agregado disponible
df = pl.read_parquet(curated_files[0])

# Aceptamos los dos nombres posibles de la columna objetivo
if "ponderado" not in df.columns and "suma_ponderada" in df.columns:
    df = df.rename({"suma_ponderada": "ponderado"})

# Variables de entrada: las que existan entre mes y anio
feature_cols = [c for c in ("mes", "anio") if c in df.columns]
X = df.select(feature_cols).to_numpy()   # entradas
y = df["ponderado"].to_numpy()           # objetivo: empleo ponderado

print("Variables de entrada:", feature_cols)
print("Filas para modelar:", len(y))

### Paso 2.2. Separar en entrenamiento, validación y prueba

Como en el Repaso, hacemos la división en tres. Reservamos la **prueba** para el final (Parte 3) y usamos la **validación** para evaluar cada modelo.

In [ ]:
# 1) apartamos la prueba (20 %); 2) del resto, una parte para validación (25 %)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42)

print("Entrenamiento:", len(y_train), "| Validación:", len(y_val), "| Prueba:", len(y_test))

### Paso 2.3. Entrenar la regresión lineal y evaluar en validación

In [ ]:
lin = LinearRegression()
lin.fit(X_train, y_train)         # aprende con el entrenamiento
y_val_lin = lin.predict(X_val)    # predice sobre validación

r2_lin = r2_score(y_val, y_val_lin)
mae_lin = mean_absolute_error(y_val, y_val_lin)
print("Lineal (validación) — R²:", round(r2_lin, 3), "| MAE:", round(mae_lin, 2))

### Paso 2.4. Graficar observado vs. predicho

Un gráfico observado-contra-predicho muestra de un vistazo qué tan bien predice: cuanto más cerca de la diagonal, mejor. **El título dice de qué capa viene** el dato (oficial GEIH): rotular la fuente es parte de la gobernanza.

In [ ]:
fig = px.scatter(
    x=y_val,
    y=y_val_lin,
    labels={"x": "Observado (empleo ponderado)", "y": "Predicho (lineal)"},
    title="Capa oficial GEIH — modelo lineal (validación)",
)
fig.write_html(str(OUTPUTS_DIR / "l6_lineal_obs_pred.html"))  # guardamos la figura
fig.show()

**Comprobar:**

In [ ]:
assert len(y_val_lin) == len(y_val)
assert (OUTPUTS_DIR / "l6_lineal_obs_pred.html").is_file()
print("Parte 2, modelo lineal: OK")

---

## Parte 3. Un modelo más flexible: red neuronal pequeña

Entrenamos una **red neuronal pequeña** (un *MLP*: perceptrón multicapa) sobre **la misma tabla** y la comparamos con la lineal. Un modelo más flexible puede ajustar mejor, pero es **más difícil de explicar** a quien decide. Esa tensión entre **precisión y explicabilidad** es central en la gobernanza de datos y a la **interpretabilidad** de los sistemas de Inteligencia Artificial.

### Paso 3.1. Escalar las entradas

Las redes neuronales aprenden mejor cuando las entradas están en una escala parecida. **`StandardScaler`** las centra y normaliza. Ajustamos el escalador **solo con el entrenamiento** (para no "mirar" la prueba). Esta acción de preprocesamiento prepara los datos para los algoritmos de aprendizaje. Una vez más estamos mejorando la calidad de nuestros datos (i.e., Assess).

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # aprende media/desviación con train
X_test_s = scaler.transform(X_test)        # aplica la misma transformación a test
print("Entradas escaladas.")

### Paso 3.2. Entrenar el MLP y evaluar en validación

Usamos **una** capa oculta con pocas neuronas: suficiente para ilustrar, sin sobreajustar una tabla pequeña. Para escalar la validación reutilizamos el `scaler` ajustado con el entrenamiento.

In [ ]:
X_val_s = scaler.transform(X_val)

mlp = MLPRegressor(hidden_layer_sizes=(8,), max_iter=500, random_state=42)
mlp.fit(X_train_s, y_train)
y_val_mlp = mlp.predict(X_val_s)

r2_mlp = r2_score(y_val, y_val_mlp)
mae_mlp = mean_absolute_error(y_val, y_val_mlp)
print("MLP (validación) — R²:", round(r2_mlp, 3), "| MAE:", round(mae_mlp, 2))

### Paso 3.3. Elegir el modelo (en validación) y evaluar en prueba

Usamos la **validación** para decidir y la **prueba** solo para la evaluación final del modelo elegido.

In [ ]:
# Elegimos por menor MAE en validación
if mae_lin <= mae_mlp:
    modelo_elegido, nombre = lin, "lineal"
    y_test_pred = modelo_elegido.predict(X_test)
else:
    modelo_elegido, nombre = mlp, "mlp"
    y_test_pred = modelo_elegido.predict(scaler.transform(X_test))

print(f"Modelo elegido (mejor en validación): {nombre}")
print("Prueba final — R²:", round(r2_score(y_test, y_test_pred), 3),
      "| MAE:", round(mean_absolute_error(y_test, y_test_pred), 2))

Aunque el MLP iguale o supere a la lineal, con la lineal podemos decir *"el empleo sube/baja tanto por mes"*; con el MLP no hay un coeficiente así de claro. Para hablarle a un ministerio, **la explicabilidad suele pesar más** que un decimal de R². Por eso elegir el modelo no es solo una cuestión de métricas.

**Comprobar:**

In [ ]:
assert len(y_test_pred) == len(y_test)
assert nombre in ("lineal", "mlp")
print("Parte 3, comparación y elección de modelo: OK")

---

## Parte 4. Monitor del stream: tasa de titulares en el tiempo

Una visualización de stream **en tiempo real** muestra cómo **cambia una métrica a medida que llegan los datos**. En un cuaderno no tenemos un flujo en vivo, pero podemos aproximarlo: tomamos los titulares que dejó la Lección 5 y graficamos su **tasa de llegada por ventana de tiempo** (una *ventana móvil*). Es una señal de **velocidad y variedad**, útil como contexto, pero **no es una medición de empleo**.

In [ ]:
news_path = STAGING_DIR / "news_labor.parquet"

if news_path.is_file():
    news = pl.read_parquet(news_path)

    # Necesitamos una marca de tiempo. Intentamos parsear 'published';
    # si no se puede, fabricamos una (un minuto entre titulares) para ilustrar el flujo.
    ts = None
    if "published" in news.columns:
        ts = news.get_column("published").str.to_datetime(strict=False)
    if ts is None or ts.null_count() == news.height:
        base = datetime(2026, 6, 20, 7, 0, tzinfo=timezone.utc)
        ts = pl.Series([base + timedelta(minutes=i) for i in range(news.height)])

    news = news.with_columns(ts.alias("ts")).drop_nulls("ts").sort("ts")

    # Ventana móvil: número de titulares por hora (tasa del stream)
    serie = (
        news.group_by_dynamic("ts", every="1h")
        .agg(pl.len().alias("titulares"))
        .sort("ts")
    )

    fig_stream = px.line(
        serie.to_pandas(),
        x="ts",
        y="titulares",
        markers=True,
        labels={"ts": "Tiempo", "titulares": "Titulares por hora"},
        title="Discurso mediático — tasa de titulares (NO es dato oficial DANE/GEIH)",
    )
    fig_stream.write_html(str(OUTPUTS_DIR / "l6_stream_rate.html"))
    fig_stream.show()
    print("Ventanas de tiempo graficadas:", serie.height)
else:
    fig_stream = None
    print("Sin news_labor.parquet — esta capa es opcional; continúe con la Parte 5.")

**Comprobar:**

In [ ]:
# Si había noticias, el gráfico debe existir
if news_path.is_file():
    assert (OUTPUTS_DIR / "l6_stream_rate.html").is_file()
print("Parte 4, monitor del stream: OK")

---

## Parte 5. Un tablero que reúne las capas de evidencia

Tener un buen modelo no basta: hay que **presentar la evidencia junta y bien rotulada**. Un **tablero** (*dashboard*) combina varias vistas en una sola figura: la **capa oficial** (el modelo sobre GEIH) y, si existe, la **capa de contexto** (la tasa del stream). Cada panel dice de dónde viene su dato — eso es gobernanza aplicada a la comunicación.

Aquí construimos un tablero con **`make_subplots`**: un panel por capa.

In [ ]:
# Un panel si no hay stream; dos si tenemos el monitor de noticias
titulos = ["Oficial GEIH: observado vs. predicho (validación)"]
if fig_stream is not None:
    titulos.append("Contexto: tasa de titulares (no oficial)")

dash = make_subplots(rows=len(titulos), cols=1, subplot_titles=titulos)

# Panel 1 — capa oficial: dispersión observado vs. predicho del modelo lineal
dash.add_trace(
    go.Scatter(x=y_val, y=y_val_lin, mode="markers", name="lineal"),
    row=1,
    col=1,
)

# Panel 2 — capa de contexto: reusamos las trazas del gráfico del stream
if fig_stream is not None:
    for traza in fig_stream.data:
        dash.add_trace(traza, row=2, col=1)

dash.update_layout(
    height=350 * len(titulos),
    showlegend=False,
    title_text="Tablero de evidencia — capa oficial (GEIH) y capa de contexto (noticias)",
)
dash.write_html(str(OUTPUTS_DIR / "l6_dashboard.html"))
dash.show()
print("Tablero guardado en:", OUTPUTS_DIR / "l6_dashboard.html")

**Cómo leerlo.** El panel oficial sostiene cualquier afirmación de empleo; el de contexto solo acompaña, y su título avisa que **no** es estadística oficial. A distintas audiencias mostraríamos distintos paneles (a quien decide, solo el oficial con su incertidumbre; a un analista, ambos).

**Comprobar:**

In [ ]:
assert (OUTPUTS_DIR / "l6_dashboard.html").is_file()
print("Parte 5, tablero de evidencia: OK")

---

## Tareas

Lo que practicaron aquí lo **implementan ustedes** en el cuaderno grupal **`week-3-group` (Ejercicio 2)**, sobre los agregados que construyan en el Ejercicio 1:

1. Dividan en **entrenamiento/validación/prueba** y entrenen el modelo elegido (lineal o MLP).
2. Hagan al menos **dos gráficos** ligados a la pregunta de decisión, con la fuente rotulada.
3. Armen un **tablero** que reúna las vistas.

**Entrega grupal:** martes **23 jun 2026** — ZIP con `week-3-group-<group_id>.ipynb` + `manifest.json`.